# Kustomize 03: generators, components, replacements

Generators build ConfigMaps and Secrets from files or literals and suffix their names with a content hash, so a changed config rolls the Deployment. Components package optional features; replacements copy a value from one resource into another.


In [ ]:
cd /source/work/kustomize-lab/base
cat > config.toml <<'TOML'
motd = "rendered with kustomize"
TOML
kustomize edit add configmap web-config --from-file=config.toml --from-literal=WHOAMI_NAME=web && kustomize build . | yq 'select(.kind == "ConfigMap") | .metadata.name, .data'


A component is a reusable, optional overlay fragment: turn on an ExternalSecret only where an environment wants it.


In [ ]:
cd /source/work/kustomize-lab
mkdir -p components/external-secret && cat > components/external-secret/externalsecret.yaml <<'YAML'
apiVersion: external-secrets.io/v1
kind: ExternalSecret
metadata:
  name: web-db
spec:
  refreshInterval: 1h
  secretStoreRef: {name: vault, kind: ClusterSecretStore}
  target: {name: web-db}
  data:
    - secretKey: DB_PASSWORD
      remoteRef: {key: web/db, property: password}
YAML
cat > components/external-secret/kustomization.yaml <<'YAML'
apiVersion: kustomize.config.k8s.io/v1alpha1
kind: Component
resources:
  - externalsecret.yaml
YAML
cd overlays/prod && kustomize edit add component ../../components/external-secret && kustomize build . | grep -c '^kind:' && kustomize build ../dev | grep -c '^kind:'


In [ ]:
cd /source/work/kustomize-lab/overlays/prod
cat >> kustomization.yaml <<'YAML'
replacements:
  - source:
      kind: Service
      name: web
      fieldPath: metadata.name
    targets:
      - select: {kind: ExternalSecret, name: web-db}
        fieldPaths: [spec.data.0.remoteRef.key]
        options: {delimiter: "/", index: 0}
YAML
kustomize build . | yq 'select(.kind == "ExternalSecret") | .spec.data[0].remoteRef.key'
